# NUMT Pipeline — Analyse interactive

Pipeline séquentiel avec vérification visuelle à chaque étape.

**Étapes :**
1. Chargement des données ClinVar
2. Extraction de séquences par regex (HGVS) — **point de contrôle principal**
3. BLAST vs génome mitochondrial (rCRS)
4. BLAST vs génome humain complet
5. Classification NUMT finale

> Modifie les paramètres dans les cellules de configuration avant de relancer.

In [ ]:
from pathlib import Path
import re
import subprocess
import sys
import time

import pandas as pd
import numpy as np

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 50)

# ── Chemins projet ────────────────────────────────────────────────────────────
PROJECT_ROOT = Path("../../").resolve()
DATA_PROCESSED = PROJECT_ROOT / "02_data" / "processed"
DATA_RAW       = PROJECT_ROOT / "02_data" / "raw"

print(f"Racine projet : {PROJECT_ROOT}")
print(f"Données processed : {DATA_PROCESSED}")
print("Imports OK")

---
## Étape 1 — Chargement des données ClinVar

Fichier produit par `01_clinvar_fetch.py` : insertions >= 50 bp toutes significations cliniques confondues.

In [ ]:
CLINVAR_TSV = DATA_PROCESSED / "clinvar_insertions_gt50bp.tsv"

df_raw = pd.read_csv(CLINVAR_TSV, sep="\t", dtype=str, low_memory=False)
print(f"Lignes totales : {len(df_raw):,}")
print(f"Colonnes : {len(df_raw.columns)}")
print()

# Assemblies présents
if 'Assembly' in df_raw.columns:
    print("Répartition par assembly :")
    print(df_raw['Assembly'].value_counts().to_string())
    print()

# Signification clinique
if 'ClinicalSignificance' in df_raw.columns:
    print("Signification clinique :")
    print(df_raw['ClinicalSignificance'].value_counts().head(10).to_string())
    print()

# Longueurs
if 'Length_bp' in df_raw.columns:
    df_raw['Length_bp'] = pd.to_numeric(df_raw['Length_bp'], errors='coerce')
    print(f"Longueur insertions : min={df_raw['Length_bp'].min():.0f}, "
          f"max={df_raw['Length_bp'].max():.0f}, "
          f"médiane={df_raw['Length_bp'].median():.0f}")

df_raw.head(5)

In [ ]:
# Dédoublonnage GRCh37/GRCh38 : garder GRCh38 si disponible
allele_col = '#AlleleID' if '#AlleleID' in df_raw.columns else 'AlleleID'

if 'Assembly' in df_raw.columns:
    df38 = df_raw[df_raw['Assembly'] == 'GRCh38'].copy()
    df37_only = df_raw[
        (df_raw['Assembly'] == 'GRCh37') &
        (~df_raw[allele_col].isin(df38[allele_col]))
    ].copy()
    df = pd.concat([df38, df37_only], ignore_index=True)
else:
    df = df_raw.copy()

print(f"Variants uniques (après dédup assembly) : {len(df):,}")
df[[allele_col, 'GeneSymbol', 'ClinicalSignificance', 'Name', 'Length_bp']].head(10)

---
## Étape 2 — Extraction des séquences par regex (HGVS)

**Formats reconnus :**
- `DIRECT_SEQUENCE` : `insATCGATCG...` — séquence littérale
- `DIRECT_MITO_REF` : `ins(NC_012920.1:m.1-72)` — référence explicite au génome mt
- `COMPLEX` : contient `inv` ou brackets avec segments non-nucléotidiques
- `LENGTH_ONLY` : `ins50` — longueur connue mais séquence non soumise
- `UNPARSEABLE` : aucun pattern reconnu

> **Modifie les regex ci-dessous si besoin, puis relance la cellule.**

In [ ]:
# ── Patterns HGVS — modifiables ───────────────────────────────────────────────

# Référence explicite au génome mitochondrial : ins(NC_012920.1:m.1-72)
RE_MITO_REF = re.compile(r"ins\((NC_012920\.\d+):m\.(\d+)[-_](\d+)\)")

# Inversion dans l'insertion
RE_INV = re.compile(r"ins.*inv")

# Brackets : ins[ATCG;GCTA]
RE_BRACKET = re.compile(r"ins\[([^\]]+)\]")

# Séquence nucléotidique directe : insATCG...
RE_DIRECT = re.compile(r"ins([ACGTNacgtn]+)$")

# Longueur seule : ins50
RE_LENGTH = re.compile(r"ins(\d+)$")


def parse_hgvs(name: str) -> dict:
    """Applique les regex dans l'ordre de spécificité."""
    result = {
        "extracted_sequence": None,
        "extraction_status": "UNPARSEABLE",
        "seq_len": None,
        "mito_ref_range": None,
    }
    if not isinstance(name, str) or not name.strip():
        return result

    # 1. Référence mitochondriale explicite
    m = RE_MITO_REF.search(name)
    if m:
        start, end = int(m.group(2)), int(m.group(3))
        result["extraction_status"] = "DIRECT_MITO_REF"
        result["mito_ref_range"] = f"{start}-{end}"
        result["seq_len"] = abs(end - start) + 1
        return result

    # 2. Inversion
    if RE_INV.search(name):
        result["extraction_status"] = "COMPLEX"
        return result

    # 3. Brackets
    m = RE_BRACKET.search(name)
    if m:
        segs = [s.strip() for s in m.group(1).split(";")]
        nuc_segs = [s.upper() for s in segs if re.fullmatch(r"[ACGTNacgtn]+", s)]
        if nuc_segs and len(nuc_segs) == len(segs):
            seq = "".join(nuc_segs)
            result.update({"extracted_sequence": seq, "extraction_status": "DIRECT_SEQUENCE", "seq_len": len(seq)})
        else:
            result["extraction_status"] = "COMPLEX"
        return result

    # 4. Séquence directe
    m = RE_DIRECT.search(name)
    if m:
        seq = m.group(1).upper()
        result.update({"extracted_sequence": seq, "extraction_status": "DIRECT_SEQUENCE", "seq_len": len(seq)})
        return result

    # 5. Longueur seule
    m = RE_LENGTH.search(name)
    if m:
        result.update({"extraction_status": "LENGTH_ONLY", "seq_len": int(m.group(1))})
        return result

    return result


print("Patterns compilés :")
print(f"  RE_MITO_REF : {RE_MITO_REF.pattern}")
print(f"  RE_DIRECT   : {RE_DIRECT.pattern}")
print(f"  RE_LENGTH   : {RE_LENGTH.pattern}")

In [ ]:
# Appliquer le parser sur toutes les lignes
parsed = df['Name'].apply(parse_hgvs).apply(pd.Series)
df = pd.concat([df, parsed], axis=1)

print("=== Résultats extraction HGVS ===")
counts = df['extraction_status'].value_counts()
print(counts.to_string())
print(f"\nTotal : {len(df):,} variants")
print(f"Avec séquence exploitable : {(df['extraction_status'].isin(['DIRECT_SEQUENCE','DIRECT_MITO_REF'])).sum():,}")

In [ ]:
# ── Inspection des UNPARSEABLE ────────────────────────────────────────────────
# Chercher dans les UNPARSEABLE des séquences mitochondriales non détectées.

unp = df[df['extraction_status'] == 'UNPARSEABLE'].copy()
print(f"UNPARSEABLE : {len(unp)} variants")
print()

# Y a-t-il des références mitochondriales cachées dans les UNPARSEABLE ?
mito_in_unp = unp[unp['Name'].str.contains('NC_012920|chrM|mitoch', case=False, na=False)]
print(f"UNPARSEABLE avec mention mitochondriale dans le Name : {len(mito_in_unp)}")
if len(mito_in_unp):
    display(mito_in_unp[[allele_col, 'GeneSymbol', 'Name', 'ClinicalSignificance']].head(20))

print()
print("=== 30 premiers noms UNPARSEABLE (pour contrôle visuel) ===")
display(unp[['Name']].head(30))

In [ ]:
# ── Exemples par statut pour vérification visuelle ───────────────────────────

for status in ['DIRECT_SEQUENCE', 'DIRECT_MITO_REF', 'LENGTH_ONLY', 'COMPLEX', 'UNPARSEABLE']:
    subset = df[df['extraction_status'] == status]
    if len(subset) == 0:
        print(f"\n[{status}] — 0 variants")
        continue
    print(f"\n[{status}] — {len(subset)} variants — 5 exemples :")
    cols = [allele_col, 'GeneSymbol', 'ClinicalSignificance', 'Name']
    if status == 'DIRECT_SEQUENCE':
        cols += ['extracted_sequence', 'seq_len']
    if status == 'DIRECT_MITO_REF':
        cols += ['mito_ref_range', 'seq_len']
    display(subset[cols].head(5))

### Étape 2b — Récupérer les séquences pour DIRECT_MITO_REF

Pour les variantes `DIRECT_MITO_REF`, le Name indique la région du génome mitochondrial (ex : `m.1-72`) mais ne donne pas la séquence elle-même. On la récupère depuis le fichier rCRS local.

In [ ]:
import urllib.request

RCRS_FASTA = DATA_PROCESSED / "rcrs_NC_012920.fasta"
RCRS_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nuccore&id=NC_012920.1&rettype=fasta&retmode=text"

if not RCRS_FASTA.exists():
    print("Téléchargement rCRS (NC_012920.1)...")
    urllib.request.urlretrieve(RCRS_URL, RCRS_FASTA)
    print(f"Sauvegardé : {RCRS_FASTA}")
else:
    print(f"rCRS déjà présent : {RCRS_FASTA}")

# Lire la séquence rCRS
rcrs_seq = ""
with open(RCRS_FASTA) as fh:
    for line in fh:
        if not line.startswith(">"):
            rcrs_seq += line.strip().upper()
print(f"Longueur rCRS : {len(rcrs_seq):,} bp (attendu ~16569)")


def fetch_mito_subseq(mito_range: str) -> str | None:
    """Retourne la sous-séquence rCRS pour un range 'start-end' (1-based)."""
    if not mito_range or not isinstance(mito_range, str):
        return None
    try:
        start, end = [int(x) for x in mito_range.split("-")]
        # Convertir en 0-based
        return rcrs_seq[start - 1 : end]
    except Exception:
        return None


# Peupler extracted_sequence pour les DIRECT_MITO_REF
mito_ref_mask = df['extraction_status'] == 'DIRECT_MITO_REF'
n_mito_ref = mito_ref_mask.sum()
print(f"\nVariants DIRECT_MITO_REF : {n_mito_ref}")

if n_mito_ref > 0:
    df.loc[mito_ref_mask, 'extracted_sequence'] = df.loc[mito_ref_mask, 'mito_ref_range'].apply(fetch_mito_subseq)
    print("Séquences rCRS injectées pour DIRECT_MITO_REF.")
    display(df[mito_ref_mask][[allele_col, 'GeneSymbol', 'mito_ref_range', 'extracted_sequence']].head(10))

### Étape 2c — Écriture du FASTA d'entrée pour BLAST

In [ ]:
QUERY_FASTA = DATA_PROCESSED / "blast_query_notebook.fasta"

# Garder uniquement les variants avec une séquence extraite
df_seq = df[df['extracted_sequence'].notna() & (df['extracted_sequence'] != '')].copy()
print(f"Variants avec séquence exploitable : {len(df_seq):,}")
print(f"  DIRECT_SEQUENCE  : {(df_seq['extraction_status']=='DIRECT_SEQUENCE').sum()}")
print(f"  DIRECT_MITO_REF  : {(df_seq['extraction_status']=='DIRECT_MITO_REF').sum()}")
print()

# Dédoublonner par AlleleID (un seul allele peut avoir GRCh37 + GRCh38)
df_seq = df_seq.drop_duplicates(subset=[allele_col])
print(f"Variants uniques pour BLAST : {len(df_seq):,}")

# Écrire le FASTA
with open(QUERY_FASTA, 'w') as fh:
    for _, row in df_seq.iterrows():
        aid  = row[allele_col]
        gene = row.get('GeneSymbol', 'NA')
        clinsig = str(row.get('ClinicalSignificance', 'NA')).replace(' ', '_')
        status = row['extraction_status']
        seq  = row['extracted_sequence']
        fh.write(f">{aid}|{gene}|{clinsig}|{status}\n{seq}\n")

print(f"FASTA écrit : {QUERY_FASTA}")

---
## Étape 3 — BLAST vs génome mitochondrial (rCRS)

> **Paramètres ajustables ci-dessous.** La cellule de résultats affiche ensuite les hits pour vérification.

In [ ]:
# ── Paramètres BLAST mito — modifiables ──────────────────────────────────────
EVALUE_MITO      = 1e-3    # Seuil E-value
WORD_SIZE_MITO   = 7       # Taille minimale du seed (7 = sensible pour courtes séq.)
PERC_ID_MITO     = 80.0    # % identité minimum (80 = tolère NUMTs anciens)
QCOV_MITO        = 50.0    # % couverture query minimum
THREADS          = 4

# Chemins
MITO_DB          = DATA_PROCESSED / "rcrs_blast_db" / "rcrs"
BLAST_MITO_OUT   = DATA_PROCESSED / "blast_mito_notebook.tsv"

print(f"Paramètres BLAST mito :")
print(f"  evalue        = {EVALUE_MITO}")
print(f"  word_size     = {WORD_SIZE_MITO}")
print(f"  perc_identity = {PERC_ID_MITO}")
print(f"  qcov_hsp_perc = {QCOV_MITO}")

In [ ]:
import shutil

# Vérifier que blastn est disponible
if not shutil.which("blastn"):
    print("ERREUR : blastn introuvable. Vérifie ton PATH.")
    print("  conda activate <env>  puis relance.")
    raise SystemExit

# Construire la base BLAST rCRS si nécessaire
MITO_DB.parent.mkdir(parents=True, exist_ok=True)
if not MITO_DB.with_suffix('.nhr').exists() and not MITO_DB.with_suffix('.nin').exists():
    print("Construction de la base BLAST rCRS...")
    subprocess.run(
        ["makeblastdb", "-in", str(RCRS_FASTA), "-dbtype", "nucl", "-out", str(MITO_DB)],
        check=True
    )
    print("Base construite.")
else:
    print(f"Base BLAST rCRS existante : {MITO_DB}")

# Lancer BLAST
print(f"\nBLAST mito en cours ({len(df_seq)} séquences)...")
t0 = time.time()

outfmt = "6 qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore qlen slen"

# Sélection du task : blastn-short si toutes les séquences < 50 bp, sinon blastn
median_len = df_seq['seq_len'].median()
task = "blastn-short" if median_len < 50 else "blastn"
print(f"  Longueur médiane queries : {median_len:.0f} bp → task={task}")

result = subprocess.run(
    [
        "blastn",
        "-query",       str(QUERY_FASTA),
        "-db",          str(MITO_DB),
        "-out",         str(BLAST_MITO_OUT),
        "-outfmt",      outfmt,
        "-evalue",      str(EVALUE_MITO),
        "-word_size",   str(WORD_SIZE_MITO),
        "-perc_identity", str(PERC_ID_MITO),
        "-qcov_hsp_perc", str(QCOV_MITO),
        "-task",        task,
        "-num_threads", str(THREADS),
    ],
    capture_output=True, text=True
)

elapsed = time.time() - t0
if result.returncode != 0:
    print(f"BLAST ERREUR:\n{result.stderr}")
else:
    print(f"BLAST terminé en {elapsed:.1f}s")
    print(f"Résultats : {BLAST_MITO_OUT}")

In [ ]:
BLAST_COLS = [
    'qseqid', 'sseqid', 'pident', 'length', 'mismatch', 'gapopen',
    'qstart', 'qend', 'sstart', 'send', 'evalue', 'bitscore', 'qlen', 'slen'
]

if BLAST_MITO_OUT.exists() and BLAST_MITO_OUT.stat().st_size > 0:
    df_mito = pd.read_csv(BLAST_MITO_OUT, sep='\t', header=None, names=BLAST_COLS)
    
    # Parser le qseqid pour récupérer AlleleID, gene, clinsig
    parts = df_mito['qseqid'].str.split('|', expand=True)
    df_mito['allele_id']   = parts[0]
    df_mito['gene']        = parts[1] if parts.shape[1] > 1 else 'NA'
    df_mito['clinsig']     = parts[2] if parts.shape[1] > 2 else 'NA'
    df_mito['ext_status']  = parts[3] if parts.shape[1] > 3 else 'NA'
    
    # Score combiné : pident * (length / qlen)
    df_mito['combined_score'] = df_mito['pident'] * (df_mito['length'] / df_mito['qlen'])
    
    # Meilleur hit par variant
    best = df_mito.sort_values('combined_score', ascending=False).drop_duplicates('allele_id')
    
    print(f"=== BLAST mito : {df_mito['allele_id'].nunique()} variants avec hit ===")
    print()
    print(f"Répartition statut extraction des hits :")
    print(df_mito.drop_duplicates('allele_id')['ext_status'].value_counts().to_string())
    print()
    print(f"Répartition signification clinique des hits :")
    print(best['clinsig'].value_counts().to_string())
    print()
    
    # Seuils de confiance
    def tier(row):
        if row['pident'] >= 90 and row['length'] >= 50 and row['evalue'] <= 1e-5:
            return 'STRONG'
        elif row['pident'] >= 80 and row['length'] >= 40:
            return 'MODERATE'
        return 'WEAK'
    
    best = best.copy()
    best['tier'] = best.apply(tier, axis=1)
    print("Tiers de confiance :")
    print(best['tier'].value_counts().to_string())
    print()
    print("Meilleur hit par variant :")
    display(best[['allele_id','gene','clinsig','pident','length','evalue','combined_score','tier']]
            .sort_values('combined_score', ascending=False))
    
    # Sauvegarder les AlleleIDs positifs
    mito_positive_ids = set(df_mito['allele_id'].astype(str))
    print(f"\nAlleleIDs mito-positifs : {len(mito_positive_ids)}")

else:
    print("Aucun résultat BLAST mito (fichier vide ou absent).")
    print("→ Vérifie la qualité des séquences dans le FASTA.")
    print("→ Essaie de relâcher les paramètres (evalue, perc_identity, word_size).")
    mito_positive_ids = set()

In [ ]:
# ── Relance avec paramètres relaxés si trop peu de hits ──────────────────────
# Décommente et ajuste si le nombre de hits est insuffisant.

RELAX_THRESHOLD = 10  # Si < N hits, proposer relaxation

if len(mito_positive_ids) < RELAX_THRESHOLD:
    print(f"Seulement {len(mito_positive_ids)} hits — relance avec paramètres relaxés...")
    print("  word_size : 7 → 4")
    print("  perc_identity : 80 → 70")
    
    BLAST_MITO_RELAXED = DATA_PROCESSED / "blast_mito_notebook_relaxed.tsv"
    
    result_r = subprocess.run(
        [
            "blastn",
            "-query",       str(QUERY_FASTA),
            "-db",          str(MITO_DB),
            "-out",         str(BLAST_MITO_RELAXED),
            "-outfmt",      outfmt,
            "-evalue",      str(EVALUE_MITO),
            "-word_size",   "4",
            "-perc_identity", "70",
            "-qcov_hsp_perc", str(QCOV_MITO),
            "-task",        task,
            "-num_threads", str(THREADS),
        ],
        capture_output=True, text=True
    )
    
    if result_r.returncode == 0 and BLAST_MITO_RELAXED.stat().st_size > 0:
        df_r = pd.read_csv(BLAST_MITO_RELAXED, sep='\t', header=None, names=BLAST_COLS)
        new_hits = df_r['qseqid'].str.split('|').str[0].nunique()
        print(f"Résultats relaxés : {new_hits} variants avec hit")
        print("→ Si le nombre est meilleur, remplace df_mito par df_r dans la cellule suivante.")
    else:
        print("Relaxation n'a pas amélioré les résultats.")
else:
    print(f"{len(mito_positive_ids)} hits — pas de relaxation nécessaire.")

---
## Étape 4 — BLAST vs génome humain (hg38)

**But :** distinguer les NUMTs de novo (absents du génome de référence) des NUMTs anciens (déjà présents dans hg38).

- `NOVEL_NUMT_CANDIDATE` : mito+ / génome– (séquence non trouvée dans le nucléaire)
- `KNOWN_REFERENCE_NUMT` : mito+ / génome+ (séquence déjà dans le génome de référence)
- `CONFIRMED_NUMT` : mito+ / génome+ au locus ClinVar rapporté

Deux modes disponibles : **local** (rapide, nécessite la base hg38) ou **remote** via NCBI (lent).

In [ ]:
# ── Paramètres BLAST génome — modifiables ─────────────────────────────────────
BLAST_GENOME_MODE   = "remote"   # "local" ou "remote"
LOCAL_GENOME_DB     = ""         # Chemin vers la base locale si mode=local
EVALUE_GENOME       = 1e-10      # Plus stringent pour le génome entier (évite les faux positifs)
PERC_ID_GENOME      = 90.0       # % identité minimum
MAX_HITS_PER_QUERY  = 20         # Nombre max de hits retournés
OVERLAP_WINDOW      = 500        # Fenêtre (bp) pour considérer qu'un hit chevauche le locus ClinVar

# Préparer le FASTA des séquences mito-positives seulement
GENOME_QUERY_FASTA  = DATA_PROCESSED / "blast_genome_query_notebook.fasta"
BLAST_GENOME_DIR    = DATA_PROCESSED / "blast_genome_notebook_xml"
BLAST_GENOME_DIR.mkdir(parents=True, exist_ok=True)

# Filtrer le FASTA sur les variants mito-positifs
df_mito_pos = df_seq[df_seq[allele_col].astype(str).isin(mito_positive_ids)].copy()
print(f"Variants mito-positifs à soumettre au BLAST génome : {len(df_mito_pos)}")

with open(GENOME_QUERY_FASTA, 'w') as fh:
    for _, row in df_mito_pos.iterrows():
        aid     = row[allele_col]
        gene    = row.get('GeneSymbol', 'NA')
        clinsig = str(row.get('ClinicalSignificance', 'NA')).replace(' ', '_')
        chrom   = str(row.get('Chromosome', 'NA'))
        start   = str(row.get('Start', 'NA'))
        stop    = str(row.get('Stop',  'NA'))
        seq     = row['extracted_sequence']
        fh.write(f">{aid}|{gene}|{clinsig}|{chrom}|{start}|{stop}\n{seq}\n")

print(f"FASTA génome écrit : {GENOME_QUERY_FASTA}")

In [ ]:
from Bio.Blast import NCBIWWW, NCBIXML

NCBI_SLEEP = 12  # secondes entre requêtes (politique NCBI)

genome_xml_results = {}  # allele_id -> liste de hits parsés

if BLAST_GENOME_MODE == "local" and LOCAL_GENOME_DB:
    # ── Mode local ────────────────────────────────────────────────────────
    BLAST_GENOME_TSV = DATA_PROCESSED / "blast_genome_notebook.tsv"
    res = subprocess.run(
        [
            "blastn",
            "-query",         str(GENOME_QUERY_FASTA),
            "-db",            LOCAL_GENOME_DB,
            "-out",           str(BLAST_GENOME_TSV),
            "-outfmt",        outfmt,
            "-evalue",        str(EVALUE_GENOME),
            "-perc_identity", str(PERC_ID_GENOME),
            "-max_target_seqs", str(MAX_HITS_PER_QUERY),
            "-num_threads",   str(THREADS),
        ],
        capture_output=True, text=True
    )
    if res.returncode != 0:
        print(f"BLAST ERREUR : {res.stderr}")
    else:
        print(f"BLAST local terminé → {BLAST_GENOME_TSV}")

else:
    # ── Mode remote (NCBI) ───────────────────────────────────────────────
    print(f"Mode remote NCBI — {len(df_mito_pos)} requêtes (~{len(df_mito_pos)*NCBI_SLEEP//60} min)")
    print("Les résultats XML sont mis en cache par AlleleID pour permettre la reprise.")
    print()

    for i, (_, row) in enumerate(df_mito_pos.iterrows(), 1):
        aid = str(row[allele_col])
        xml_cache = BLAST_GENOME_DIR / f"{aid}.xml"

        if xml_cache.exists() and xml_cache.stat().st_size > 100:
            print(f"  [{i}/{len(df_mito_pos)}] {aid} — cache OK")
            continue

        seq = row['extracted_sequence']
        print(f"  [{i}/{len(df_mito_pos)}] {aid} — BLAST en cours...", end=" ", flush=True)
        try:
            handle = NCBIWWW.qblast(
                "blastn", "nt", seq,
                entrez_query="Homo sapiens[ORGN]",
                expect=EVALUE_GENOME,
                perc_ident=int(PERC_ID_GENOME),
                hitlist_size=MAX_HITS_PER_QUERY,
            )
            with open(xml_cache, 'w') as fh:
                fh.write(handle.read())
            print("OK")
        except Exception as exc:
            print(f"ERREUR : {exc}")

        if i < len(df_mito_pos):
            time.sleep(NCBI_SLEEP)

    print("\nToutes les requêtes terminées (ou en cache).")

In [ ]:
# ── Table de correspondance accession → chr ───────────────────────────────────
ACCESSION_TO_CHR = {
    "NC_000001.11": "chr1",  "NC_000002.12": "chr2",  "NC_000003.12": "chr3",
    "NC_000004.12": "chr4",  "NC_000005.10": "chr5",  "NC_000006.12": "chr6",
    "NC_000007.14": "chr7",  "NC_000008.11": "chr8",  "NC_000009.12": "chr9",
    "NC_000010.11": "chr10", "NC_000011.10": "chr11", "NC_000012.12": "chr12",
    "NC_000013.11": "chr13", "NC_000014.9":  "chr14", "NC_000015.10": "chr15",
    "NC_000016.10": "chr16", "NC_000017.11": "chr17", "NC_000018.10": "chr18",
    "NC_000019.10": "chr19", "NC_000020.11": "chr20", "NC_000021.9":  "chr21",
    "NC_000022.11": "chr22", "NC_000023.11": "chrX",  "NC_000024.10": "chrY",
    "NC_012920.1":  "chrM",
}
CANONICAL_CHRS = set(ACCESSION_TO_CHR.values())  # chr1..chr22, chrX, chrY, chrM


def normalise_chrom(sid: str) -> str:
    """Normalise un subject_id BLAST en nom de chromosome standard."""
    if sid.startswith("chr"):
        return sid
    if sid in ACCESSION_TO_CHR:
        return ACCESSION_TO_CHR[sid]
    # Format gi|...|ref|NC_XXXXXX.N|
    m = re.search(r"(NC_\d+\.\d+)", sid)
    if m and m.group(1) in ACCESSION_TO_CHR:
        return ACCESSION_TO_CHR[m.group(1)]
    # NW_/NT_ = loci alternatifs/patches → pas un chromosome canonique
    return sid  # gardé tel quel, sera traité comme non-nucléaire


def is_nuclear(chrom: str) -> bool:
    """True si le chromosome est un chromosome nucléaire canonique (chr1–22, X, Y)."""
    return chrom in CANONICAL_CHRS and chrom not in ("chrM", "chrMT")


def is_mito(chrom: str) -> bool:
    return chrom in ("chrM", "chrMT", "NC_012920.1")


# ── Parser les XML BLAST génome ───────────────────────────────────────────────
genome_hits = {}  # allele_id -> list of hit dicts
unknown_accessions = set()

xml_files = list(BLAST_GENOME_DIR.glob("*.xml"))
print(f"Fichiers XML trouvés : {len(xml_files)}")

for xml_file in xml_files:
    aid = xml_file.stem
    try:
        with open(xml_file) as fh:
            records = list(NCBIXML.parse(fh))
    except Exception as exc:
        print(f"  ERREUR parsing {xml_file.name}: {exc}")
        continue

    hits = []
    for record in records:
        for alignment in record.alignments:
            raw_sid = alignment.title.split()[0]
            chrom = normalise_chrom(raw_sid)
            if chrom == raw_sid and not raw_sid.startswith("chr"):
                # Accession non reconnue
                unknown_accessions.add(raw_sid[:30])
            for hsp in alignment.hsps:
                hits.append({
                    "allele_id":   aid,
                    "raw_subject": raw_sid,
                    "chromosome":  chrom,
                    "hit_start":   hsp.sbjct_start,
                    "hit_end":     hsp.sbjct_end,
                    "pident":      hsp.identities / hsp.align_length * 100,
                    "length":      hsp.align_length,
                    "evalue":      hsp.expect,
                    "is_nuclear":  is_nuclear(chrom),
                    "is_mito":     is_mito(chrom),
                })
    genome_hits[aid] = hits

print(f"Variants parsés : {len(genome_hits)}")
print(f"Variants avec au moins 1 hit : {sum(1 for h in genome_hits.values() if h)}")

if unknown_accessions:
    print(f"\n⚠ Accessions non reconnues ({len(unknown_accessions)}) — loci alternatifs / patches :")
    for a in sorted(unknown_accessions)[:15]:
        print(f"  {a}")
    print("  → Ces hits sont ignorés dans la classification (non-nucléaires canoniques).")

In [ ]:
# ── Inspecter la distribution des chromosomes touchés ────────────────────────
all_hits_flat = [h for hits in genome_hits.values() for h in hits]

if all_hits_flat:
    df_ghits = pd.DataFrame(all_hits_flat)
    print("Distribution des chromosomes dans les hits génome :")
    chrom_counts = df_ghits['chromosome'].value_counts()
    display(chrom_counts.head(30).to_frame())
    print()
    print(f"Hits nucléaires canoniques : {df_ghits['is_nuclear'].sum()}")
    print(f"Hits mitochondriaux        : {df_ghits['is_mito'].sum()}")
    print(f"Hits non classifiés        : {(~df_ghits['is_nuclear'] & ~df_ghits['is_mito']).sum()}")
else:
    print("Aucun hit génome à analyser.")

---
## Étape 5 — Classification NUMT

| Classification | Critère |
|---|---|
| `CONFIRMED_NUMT` | mito+ ET hit nucléaire au locus ClinVar rapporté |
| `KNOWN_REFERENCE_NUMT` | mito+ ET hit nucléaire ailleurs (NUMT ancestral) |
| `NOVEL_NUMT_CANDIDATE` | mito+ ET aucun hit nucléaire canonique (de novo) |
| `AMBIGUOUS` | hit nucléaire sans bon match mito |

In [ ]:
rows = []

for _, row in df_mito_pos.iterrows():
    aid = str(row[allele_col])

    # Infos ClinVar
    gene    = row.get('GeneSymbol', 'NA')
    clinsig = row.get('ClinicalSignificance', 'NA')
    chrom_cv = str(row.get('Chromosome', ''))
    if chrom_cv and not chrom_cv.startswith('chr'):
        chrom_cv = f"chr{chrom_cv}"
    try:
        start_cv = int(row.get('Start', 0) or 0)
        stop_cv  = int(row.get('Stop',  0) or 0)
    except (ValueError, TypeError):
        start_cv = stop_cv = 0

    # Infos BLAST mito (step 3)
    best_mito = df_mito[df_mito['allele_id'] == aid].sort_values('combined_score', ascending=False)
    has_mito_hit = len(best_mito) > 0
    best_pident  = float(best_mito.iloc[0]['pident']) if has_mito_hit else None

    # Infos BLAST génome (step 4)
    hits = genome_hits.get(aid, [])
    nuclear_hits = [h for h in hits if h['is_nuclear']]
    has_nuclear = len(nuclear_hits) > 0
    nuclear_chroms = list({h['chromosome'] for h in nuclear_hits})

    # Vérifier chevauchement avec le locus ClinVar
    overlaps_clinvar = False
    for h in nuclear_hits:
        if h['chromosome'] == chrom_cv and start_cv > 0:
            hmin = min(h['hit_start'], h['hit_end'])
            hmax = max(h['hit_start'], h['hit_end'])
            cv_lo = start_cv - OVERLAP_WINDOW
            cv_hi = (stop_cv or start_cv) + OVERLAP_WINDOW
            if hmin <= cv_hi and hmax >= cv_lo:
                overlaps_clinvar = True
                break

    # Classification
    if has_mito_hit and overlaps_clinvar:
        classification = "CONFIRMED_NUMT"
    elif has_nuclear and not overlaps_clinvar:
        classification = "KNOWN_REFERENCE_NUMT"
    elif has_mito_hit and not has_nuclear:
        classification = "NOVEL_NUMT_CANDIDATE"
    elif has_nuclear and not has_mito_hit:
        classification = "AMBIGUOUS"
    else:
        classification = "NOVEL_NUMT_CANDIDATE"  # mito+ sans hit génome

    rows.append({
        'AlleleID':          aid,
        'Gene':              gene,
        'ClinSig':           clinsig,
        'ExtStatus':         row.get('extraction_status', 'NA'),
        'Classification':    classification,
        'BestMitoPident':    best_pident,
        'HasNuclearHit':     has_nuclear,
        'OverlapsClinVar':   overlaps_clinvar,
        'NuclearChroms':     ', '.join(nuclear_chroms) if nuclear_chroms else 'none',
        'NumGenomeHits':     len(hits),
        'NumNuclearHits':    len(nuclear_hits),
    })

df_final = pd.DataFrame(rows)

print("=== Classification finale ===")
print(df_final['Classification'].value_counts().to_string())
print()
print(f"  NUMTs de novo  (NOVEL_NUMT_CANDIDATE) : {(df_final['Classification']=='NOVEL_NUMT_CANDIDATE').sum()}")
print(f"  NUMTs anciens  (KNOWN_REFERENCE_NUMT) : {(df_final['Classification']=='KNOWN_REFERENCE_NUMT').sum()}")
print(f"  NUMTs confirmés (CONFIRMED_NUMT)       : {(df_final['Classification']=='CONFIRMED_NUMT').sum()}")
print(f"  Ambigus                                : {(df_final['Classification']=='AMBIGUOUS').sum()}")

In [ ]:
# ── Table finale avec mise en valeur des candidats ───────────────────────────
print("=== NUMTs de novo (NOVEL_NUMT_CANDIDATE) ===")
novel = df_final[df_final['Classification'] == 'NOVEL_NUMT_CANDIDATE']
display(novel[['AlleleID','Gene','ClinSig','BestMitoPident','ExtStatus']]
        .sort_values('BestMitoPident', ascending=False))

print()
print("=== NUMTs anciens (KNOWN_REFERENCE_NUMT) ===")
known = df_final[df_final['Classification'] == 'KNOWN_REFERENCE_NUMT']
display(known[['AlleleID','Gene','ClinSig','BestMitoPident','NuclearChroms','NumNuclearHits']])

print()
print("=== NUMTs confirmés (CONFIRMED_NUMT) ===")
confirmed = df_final[df_final['Classification'] == 'CONFIRMED_NUMT']
if len(confirmed):
    display(confirmed)
else:
    print("  Aucun")

# Sauvegarder la table
OUTPUT_TSV = DATA_PROCESSED / "numt_classification_notebook.tsv"
df_final.to_csv(OUTPUT_TSV, sep='\t', index=False)
print(f"\nTable sauvegardée : {OUTPUT_TSV}")